# Custom inline evalscript

You are not limited to the bundled recipes: pass a custom V3 evalscript (an inline string or a `.js` path) over a plain collection key. Here, a simple NDWI water index.

## Credentials guard

In [ ]:
import os
from pathlib import Path


def has_sh_credentials() -> bool:
    """Whether Sentinel Hub OAuth client-credentials are available."""
    return bool(os.environ.get("SH_CLIENT_ID") and os.environ.get("SH_CLIENT_SECRET"))


def run_or_skip(facade, **download_kwargs):
    """Run the live download when credentials exist, else print a skip note.

    Keeps the notebook executing top-to-bottom with no errors whether or not
    Sentinel Hub credentials are configured (so the docs build never needs
    secrets). Set SH_CLIENT_ID / SH_CLIENT_SECRET to run the cell for real.
    """
    if not has_sh_credentials():
        print(
            "No Sentinel Hub credentials found - skipping the live download.\n"
            "Mint an OAuth client_credentials pair in the CDSE Dashboard and set\n"
            "SH_CLIENT_ID / SH_CLIENT_SECRET (see the Authentication page)."
        )
        return []
    results = facade.download(**download_kwargs)
    for item in results:
        print(item)
    return results


## A custom evalscript over a plain collection

In [ ]:
from earthlens import EarthLens

NDWI = '''//VERSION=3
function setup() { return { input: ["B03", "B08"], output: { bands: 1, sampleType: "FLOAT32" } }; }
function evaluatePixel(s) { return [(s.B03 - s.B08) / (s.B03 + s.B08)]; }
'''

facade = EarthLens(
    data_source='sentinel-hub',
    variables={'sentinel-2-l2a': []},   # plain collection
    start='2020-06-10', end='2020-06-20',
    lat_lim=[40.80, 40.83], lon_lim=[14.24, 14.27],
    path='data/sh-custom', resolution=20,
    evalscript=NDWI,
)
paths = run_or_skip(facade)
paths